# Tool Use and Reflective Agents

Build a research pipeline that searches arXiv and the web, critiques its report, rewrites it, and publishes the result as styled HTML.

## Setup

In [ ]:
# Standard-library imports.
import json
from typing import Any

# Third-party and project imports.
from dotenv import load_dotenv
from IPython.display import HTML, display
from openai import OpenAI
import research_tools

# Load API keys from .env and create the OpenAI client.
load_dotenv()
client = OpenAI()

## Research tools

In [ ]:
# The research_tools module contains the provided arXiv and Tavily helpers.
# Uncomment this line to test the arXiv helper before running the full workflow.
# print(research_tools.arxiv_search_tool('retrieval-augmented generation', max_results=3))

## Tool definitions and mapping

In [ ]:
# Map model-visible tool names to their local Python implementations.
TOOL_DEFINITIONS = [research_tools.arxiv_tool_def, research_tools.tavily_tool_def]
TOOL_MAPPING = {
    'arxiv_search_tool': research_tools.arxiv_search_tool,
    'tavily_search_tool': research_tools.tavily_search_tool,
}

## Exercise 1: Tool-calling research assistant

In [ ]:
def generate_research_report_with_tools(
    prompt_: str,
    model: str = 'gpt-4o',
    max_turns: int = 10,
) -> list[dict[str, Any]]:
    # Let the model request tools until it returns a final, sourced report.
    messages: list[dict[str, Any]] = [
        {
            'role': 'system',
            'content': (
                'You are a research assistant. Use arXiv for academic evidence and web search for current context. '
                'Write a detailed, neutral report with clear sections and inline citations that include full URLs. '
                'Never invent sources or citations.'
            ),
        },
        {'role': 'user', 'content': prompt_},
    ]

    for _ in range(max_turns):
        response = client.chat.completions.create(
            model=model,
            messages=messages,
            tools=TOOL_DEFINITIONS,
            tool_choice='auto',
            temperature=0.4,
        )
        message = response.choices[0].message
        messages.append(message.model_dump(exclude_none=True))

        if not message.tool_calls:
            return messages

        for tool_call in message.tool_calls:
            tool_name = tool_call.function.name
            try:
                arguments = json.loads(tool_call.function.arguments)
                result = TOOL_MAPPING[tool_name](**arguments)
            except Exception as error:
                result = {'error': f'{type(error).__name__}: {error}'}

            messages.append({
                'role': 'tool',
                'tool_call_id': tool_call.id,
                'name': tool_name,
                'content': json.dumps(result),
            })

    raise RuntimeError(f'The model did not complete within {max_turns} turns.')

## Exercise 2: Reflection and rewrite

In [ ]:
def extract_final_assistant_text(text_or_messages: str | list[dict[str, Any]]) -> str:
    # Accept either plain report text or the full tool-calling message history.
    if not isinstance(text_or_messages, list):
        return str(text_or_messages).strip()

    for message in reversed(text_or_messages):
        if message.get('role') == 'assistant' and message.get('content'):
            return str(message['content']).strip()
    raise ValueError('No final assistant report was found in the message history.')


def reflection_and_rewrite(
    text_or_messages: str | list[dict[str, Any]],
    model: str = 'gpt-4o-mini',
    temperature: float = 0.3,
) -> dict[str, str]:
    # Produce a concise structured critique and a separate improved report.
    report = extract_final_assistant_text(text_or_messages)
    response = client.chat.completions.create(
        model=model,
        messages=[
            {'role': 'system', 'content': 'You are a concise academic reviewer and editor.'},
            {
                'role': 'user',
                'content': (
                    'Return valid JSON with exactly two string fields: reflection and revised_report. '
                    'The reflection must use these labeled headings: Strengths, Limitations, Suggestions, Opportunities. '
                    'Preserve only well-supported claims and citations in the revised report.\n\n'
                    f'Report to review:\n{report}'
                ),
            },
        ],
        temperature=temperature,
        response_format={
            'type': 'json_schema',
            'json_schema': {
                'name': 'reflection_and_rewrite',
                'strict': True,
                'schema': {
                    'type': 'object',
                    'properties': {
                        'reflection': {'type': 'string'},
                        'revised_report': {'type': 'string'},
                    },
                    'required': ['reflection', 'revised_report'],
                    'additionalProperties': False,
                },
            },
        },
    )
    return json.loads(response.choices[0].message.content)

## Exercise 3: Publish as HTML

In [ ]:
def convert_report_to_html(text_or_messages: str | list[dict[str, Any]], model: str = 'gpt-4o-mini') -> str:
    # Turn the final report into a complete HTML document with clickable citations.
    report = extract_final_assistant_text(text_or_messages)
    response = client.chat.completions.create(
        model=model,
        messages=[
            {'role': 'system', 'content': 'You convert research reports into accessible, self-contained HTML documents.'},
            {
                'role': 'user',
                'content': (
                    'Convert this report into a complete, clean HTML document with embedded CSS, section headings, '
                    'readable paragraphs, and clickable source links. Preserve citations accurately. '
                    'Return only valid HTML, with no Markdown fences or explanation.\n\n'
                    f'Report:\n{report}'
                ),
            },
        ],
        temperature=0.2,
    )
    return response.choices[0].message.content.strip()

## End-to-end pipeline

In [ ]:
# Run the complete pipeline after adding both API keys to .env.
prompt_ = 'Radio observations of recurrent novae'

message_history = generate_research_report_with_tools(prompt_)
preliminary_report = extract_final_assistant_text(message_history)
review = reflection_and_rewrite(preliminary_report)
html_report = convert_report_to_html(review['revised_report'])

print('=== Preliminary report ===\n')
print(preliminary_report)
print('\n=== Reflection ===\n')
print(review['reflection'])
print('\n=== Revised report ===\n')
print(review['revised_report'])

# Render the completed report directly below the cell.
display(HTML(html_report))